In [1]:
import jax
import jax.numpy as jnp
import optax
from tqdm import tqdm
import numpy as np
import json

import pennylane as qml

In [2]:
import os, sys
sys.path.append('../')

from pqcqec.utils.constants import QUBITS_FOR_GATES, QISKIT_GATES, GATE_IS_DIRECTIONAL, PENNYLANE_GATES
from pqcqec.noise.simple_noise import PennylaneNoisyGates
# from pqcqec.simulate.simulate import run_circuit_with_noise_model
from pqcqec.circuits.modify import pennylane_state_embedding

In [3]:
PQC_GATES = ['rz', 'rx', 'rz']
DATA_PATH = '../nogit/circuit_tokens/no_uncomp/5q_500g_circuit_data/'
GOOD_DATA_PATH = DATA_PATH + 'per_seed_data/'
BAD_DATA_PATH = DATA_PATH + 'poor_fidelity/'
CONFIG_PATH = DATA_PATH + 'config.json'

with open(CONFIG_PATH, 'r') as f:
    CONFIG = json.load(f)


NUM_QUBITS = CONFIG.get("qubits", 3)[0]
NUM_GATES = CONFIG.get("gates", 4)[0] # Multiply by 2 for uncomp gates. 
GATE_BLOCKS = CONFIG.get("gate_blocks", 4)
VALID_GATES = CONFIG.get("gate_dist", QISKIT_GATES)
if VALID_GATES:
    VALID_GATES = list(VALID_GATES.keys())
else:
    VALID_GATES = ['x', 'z', 'h', 'cx', 'cz']

print(f"Using {NUM_QUBITS} qubits, {NUM_GATES} gates, {GATE_BLOCKS} gate blocks, {VALID_GATES} valid gates.")

NOISE_DIST = {"x_rad": 0.01, "z_rad": 0.01, "delta_x": 0, "delta_z": 0}

PAD_ID = 0
UNDIRECTED_GATES = [gate for gate in VALID_GATES if not GATE_IS_DIRECTIONAL.get(gate, False)]
print(UNDIRECTED_GATES)

TRAIN_SZ = 0.8
VAL_SZ = 0.1
TEST_SZ = 0.1
BATCH_SIZE = 64

LEARNING_RATE = 5e-6
WEIGHT_DECAY = 1e-3


Using 5 qubits, 500 gates, 4 gate blocks, ['x', 'z', 'h', 'cx', 'cz'] valid gates.
['x', 'z', 'h', 'cz']


In [4]:

good_data = []
poor_data = []

for i, filename in enumerate(os.listdir(GOOD_DATA_PATH)):
    if i > 10000:
        break
    with open(GOOD_DATA_PATH + filename, 'r') as f:
        token_dict = json.load(f)
        good_data.append(token_dict['base_circuit_tokens'])
        f.close()

# for filename in os.listdir(BAD_DATA_PATH):
#     with open(BAD_DATA_PATH + filename, 'r') as f:
#         token_dict = json.load(f)
#         poor_data.append((token_dict['base_circuit_tokens'], token_dict['pqc_params'], token_dict['fidelity']))
#         f.close()

print(f"Number of good data samples: {len(good_data)}")
# print(f"Number of poor data samples: {len(poor_data)}")

Number of good data samples: 10001


In [5]:

def simple_circuit_simulator(circuit_ops, input_state, num_qubits, x_noise, z_noise):
    """Runs a quantum circuit with a noise model using PennyLane and PyTorch."""
    qdevice = qml.device("default.qubit", wires=num_qubits)

    @qml.qnode(qdevice)
    def circuit(state):
        pennylane_state_embedding(state, num_qubits)
        for i, op in enumerate(circuit_ops):
            gate, wires, param = op
            # noise_model.apply_gate(gate, wires, angle=param)
            PENNYLANE_GATES[gate](wires=wires)
            for wire in wires:
                qml.RX(x_noise[i], wires=[wire])  # Example noise application
                qml.RZ(z_noise[i], wires=[wire])  # Example noise application

        return qml.state()

    # The function now directly returns the output of the torch interface qnode
    return circuit(input_state)

In [6]:
for data in tqdm(good_data):
    input_state = np.zeros((100, 2**NUM_QUBITS))
    input_state[:,0] = 1.0
    x_noise = np.ones(len(data)) * 0.01
    z_noise = np.ones(len(data)) * 0.01
    output_state = simple_circuit_simulator(data, input_state, NUM_QUBITS, x_noise, z_noise)
    # print(f"Output state: {output_state}")

100%|██████████| 10001/10001 [26:27<00:00,  6.30it/s] 


In [7]:
import time

input_state = np.zeros((100, 2**NUM_QUBITS))
input_state[:,0] = 1.0


x_noise = np.ones(len(good_data[0])) * 0.01
z_noise = np.ones(len(good_data[0])) * 0.01

start_time = time.time()
output_state = simple_circuit_simulator(good_data[0], input_state, NUM_QUBITS, x_noise, z_noise)
end_time = time.time()

execution_time = end_time - start_time
print(f"Execution time: {execution_time} seconds")
print(f"Output state: {output_state.shape}")


Execution time: 0.16127300262451172 seconds
Output state: (100, 32)


In [9]:
import time

input_state = np.zeros((2**NUM_QUBITS,))
input_state[0] = 1.0


x_noise = np.ones(len(good_data[10])) * 0.01
z_noise = np.ones(len(good_data[10])) * 0.01


start_time = time.time()
output_state = simple_circuit_simulator(good_data[10], input_state, NUM_QUBITS, x_noise, z_noise)
end_time = time.time()

execution_time = end_time - start_time
print(f"Execution time: {execution_time} seconds")


Execution time: 0.08584928512573242 seconds
